<a href="https://colab.research.google.com/github/SUKANYA12345678/port-folio/blob/main/finalDeepFakeDectector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# kaggle dataset link: https://www.kaggle.com/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset

In [ ]:
!pip install deepface librosa opencv-python matplotlib tensorflow mtcnn lz4
!pip install deepface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.7 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
from pathlib import Path
from collections import deque
import scipy.stats as stats
import time

class OptimalDeepfakeFrameExtractor:
    def __init__(self):
        """
        🎯 OPTIMAL DEEPFAKE FRAME EXTRACTOR

        OPTIMIZATIONS FOR DEEPFAKE DETECTION:
        - Dense temporal sampling (every 0.1-0.2s)
        - Face-priority extraction
        - Motion transition detection
        - Lighting change detection
        - Multi-scale quality assessment
        - Temporal inconsistency detection
        """
        self.frame_history = deque(maxlen=15)  # Longer history for better analysis
        self.face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

        # 🎯 DEEPFAKE-OPTIMIZED THRESHOLDS
        self.THRESHOLDS = {
            # Face-specific thresholds (most important for deepfakes)
            'face_quality_min': 30,              # Minimum face quality
            'face_consistency_threshold': 0.15,   # Face appearance consistency
            'face_size_min_pixels': 80,          # Minimum face size
            'face_blur_max': 2.5,               # Maximum acceptable face blur

            # Temporal inconsistency (deepfakes often fail here)
            'temporal_face_change_max': 0.25,    # Face should be consistent
            'lighting_change_threshold': 15,     # Sudden lighting changes suspicious
            'motion_inconsistency_max': 0.3,     # Motion should be natural
            'frame_jump_threshold': 25,          # Frame-to-frame jump detection

            # Deepfake artifacts
            'boundary_artifact_threshold': 20,   # Face boundary artifacts
            'compression_inconsistency': 0.2,    # Compression mismatch
            'color_space_anomaly': 15,           # Unnatural color transitions
            'texture_mismatch_threshold': 0.18,  # Face-background texture mismatch

            # Quality and sharpness (deepfakes often have quality issues)
            'sharpness_variation_max': 0.4,      # Sharpness should be consistent
            'contrast_inconsistency_max': 0.3,   # Contrast consistency
            'noise_pattern_anomaly': 0.25,       # Unnatural noise patterns

            # Modern deepfake signatures (2024/2025)
            'gan_artifact_threshold': 0.15,      # GAN-specific artifacts
            'diffusion_smoothness_max': 0.92,    # Over-smoothness from diffusion
            'frequency_anomaly_min': 0.12,       # Frequency domain anomalies
        }

        # Frame extraction strategy
        self.EXTRACTION_STRATEGY = {
            'base_interval': 0.15,          # Extract every 0.15 seconds (dense)
            'transition_boost': 3,          # 3x more frames during transitions
            'face_priority_boost': 2,       # 2x more frames with faces
            'quality_threshold_boost': 1.5, # 1.5x more high-quality frames
            'max_frames_per_second': 10,    # Maximum frame density
        }


        print("🎯 OPTIMAL DEEPFAKE FRAME EXTRACTOR INITIALIZED")
        print("   🎭 Face-priority extraction enabled")
        print("   ⚡ Dense temporal sampling (0.1-0.2s intervals)")
        print("   🔍 Motion transition detection active")
        print("   📊 Multi-scale quality assessment")
        print("   🎬 Optimized for modern deepfake detection")

    def extract_optimal_frames(self, video_path, max_frames=300, quality_focus="deepfake"):
        """
        🚀 Extract optimal frames for deepfake detection

        Args:
            video_path: Path to video file
            max_frames: Maximum frames to extract (300 for thorough analysis)
            quality_focus: "deepfake" or "general" (deepfake = face priority)
        """

        if not Path(video_path).exists():
            raise FileNotFoundError(f"Video file not found: {video_path}")

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Cannot open video file: {video_path}")

        # Get video properties
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / fps if fps > 0 else 0
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        print(f"🎬 OPTIMAL DEEPFAKE ANALYSIS STARTING:")
        print(f"   📊 Resolution: {width}x{height}")
        print(f"   ⏱️  Duration: {duration:.1f} seconds")
        print(f"   🎬 FPS: {fps:.1f}")
        print(f"   📝 Total Frames: {total_frames:,}")
        print(f"   🎯 Target Extraction: {max_frames} frames ({max_frames/duration:.1f}/sec)")

        # Adaptive frame skip calculation
        base_skip = max(1, int(fps * self.EXTRACTION_STRATEGY['base_interval']))
        print(f"   📈 Base sampling: every {base_skip} frames ({self.EXTRACTION_STRATEGY['base_interval']}s)")

        extracted_frames = []
        deepfake_stats = {
            'face_frames': 0,
            'transition_frames': 0,
            'high_quality_frames': 0,
            'suspicious_frames': 0,
            'temporal_inconsistencies': 0,
            'boundary_artifacts': 0,
            'total_deepfake_flags': 0,
            'extraction_reasons': {'face': 0, 'quality': 0, 'transition': 0, 'suspicious': 0, 'temporal': 0}
        }

        try:
            frame_number = 0
            extracted_count = 0
            last_extracted_frame = None

            print(f"\n🔍 Starting intelligent frame extraction...")
            start_time = time.time()

            while extracted_count < max_frames and frame_number < total_frames:
                ret, frame = cap.read()
                if not ret:
                    break

                frame_number += 1
                timestamp = frame_number / fps if fps > 0 else frame_number

                # 🎯 INTELLIGENT EXTRACTION DECISION
                extraction_decision = self._should_extract_frame_optimal(
                    frame, frame_number, base_skip, extracted_count, max_frames, timestamp, total_frames, cap
                )

                if extraction_decision['extract']:
                    # 🔬 COMPREHENSIVE DEEPFAKE ANALYSIS
                    frame_analysis = self._analyze_frame_for_deepfakes(
                        frame, frame_number, timestamp, last_extracted_frame
                    )

                    # Update statistics
                    self._update_deepfake_statistics(frame_analysis, deepfake_stats, extraction_decision['reasons'])

                    # Store with rich metadata
                    extracted_frames.append(frame_analysis)

                    # Update frame history
                    self._update_frame_history(frame_analysis)

                    last_extracted_frame = frame_analysis
                    extracted_count += 1

                    # Progress updates
                    if extracted_count % 50 == 0:
                        elapsed = time.time() - start_time
                        rate = extracted_count / elapsed
                        eta = (max_frames - extracted_count) / rate
                        print(f"   🔄 Extracted {extracted_count}/{max_frames} frames "
                              f"({rate:.1f}/sec, ETA: {eta:.1f}s)")

        finally:
            cap.release()

        print(f"   ✅ Extraction complete! {extracted_count} frames in {time.time() - start_time:.1f}s")

        # 🏆 ORGANIZE RESULTS WITH DEEPFAKE FOCUS
        result = self._organize_deepfake_results(extracted_frames, deepfake_stats, total_frames, duration, fps)

        # Print detailed analysis
        self._print_deepfake_analysis_summary(result)

        return result

    def _should_extract_frame_optimal(self, frame, frame_number, base_skip, extracted_count, max_frames, timestamp, total_frames, cap):
        """🎯 Intelligent frame extraction decision"""

        reasons = []
        extract = False

        # 1. MANDATORY EXTRACTIONS
        # Always extract first 10 and last 10 frames (boundaries are crucial)
        if frame_number <= 10:
            extract = True
            reasons.append("boundary_start")
        elif frame_number >= (total_frames - 10):
            extract = True
            reasons.append("boundary_end")

        # 2. BASE SAMPLING
        elif frame_number % base_skip == 0:
            extract = True
            reasons.append("base_sampling")

        # 3. FACE PRIORITY BOOST
        face_detected = self._detect_faces_quick(frame)
        if face_detected and frame_number % (base_skip // 2) == 0:  # 2x sampling with faces
            extract = True
            reasons.append("face_priority")

        # 4. MOTION/TRANSITION DETECTION
        if len(self.frame_history) > 0:
            motion_level = self._calculate_motion_level_fast(frame)
            if motion_level > 0.3:  # Significant motion
                extract = True
                reasons.append("motion_transition")

        # 5. QUALITY-BASED EXTRACTION
        if extracted_count < max_frames * 0.7:  # Don't fill up on quality alone
            quality_score = self._quick_quality_assessment(frame)
            if quality_score > 0.8:  # High quality frame
                extract = True
                reasons.append("high_quality")

        # 6. SPACING CONTROL - don't extract too densely
        if len(self.frame_history) > 0:
            last_frame_num = self.frame_history[-1]['frame_number']
            if frame_number - last_frame_num < max(2, base_skip // 3):
                # Too close to last extraction (unless it's a priority case)
                priority_reasons = ['face_priority', 'boundary_start', 'boundary_end']
                if not any(r in reasons for r in priority_reasons):
                    extract = False
                    reasons = ["spacing_control_skip"]

        return {
            'extract': extract,
            'reasons': reasons
        }

    def _analyze_frame_for_deepfakes(self, frame, frame_number, timestamp, last_frame):
        """🔬 Comprehensive deepfake-focused analysis"""

        # Convert to different color spaces
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)

        # 1. BASIC METRICS
        basic_metrics = self._calculate_enhanced_basic_metrics(gray, frame)

        # 2. FACE-SPECIFIC ANALYSIS
        face_metrics = self._analyze_faces_comprehensive(frame, gray)

        # 3. DEEPFAKE ARTIFACT DETECTION
        deepfake_artifacts = self._detect_deepfake_artifacts(frame, gray, hsv, lab)

        # 4. TEMPORAL ANALYSIS (if history available)
        temporal_metrics = self._analyze_temporal_consistency(frame, gray, last_frame)

        # 5. FREQUENCY DOMAIN ANALYSIS
        frequency_metrics = self._analyze_frequency_domain_detailed(gray)

        # 6. TEXTURE AND BOUNDARY ANALYSIS
        texture_metrics = self._analyze_texture_boundaries(frame, gray)

        # 7. CALCULATE DEEPFAKE SUSPICION
        deepfake_suspicion = self._calculate_deepfake_suspicion_score(
            basic_metrics, face_metrics, deepfake_artifacts,
            temporal_metrics, frequency_metrics, texture_metrics
        )

        return {
            'frame': frame.copy(),
            'metadata': {
                'frame_number': frame_number,
                'timestamp': timestamp,
                'basic_metrics': basic_metrics,
                'face_metrics': face_metrics,
                'deepfake_artifacts': deepfake_artifacts,
                'temporal_metrics': temporal_metrics,
                'frequency_metrics': frequency_metrics,
                'texture_metrics': texture_metrics,
                'deepfake_suspicion': deepfake_suspicion,
                'total_flags': sum([
                    deepfake_artifacts.get('total_flags', 0),
                    temporal_metrics.get('inconsistency_flags', 0),
                    face_metrics.get('anomaly_flags', 0)
                ])
            }
        }

    def _detect_faces_quick(self, frame):
        """Quick face detection for extraction decisions"""
        try:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = self.face_cascade.detectMultiScale(gray, 1.1, 4, minSize=(50, 50))
            return len(faces) > 0
        except:
            return False

    def _calculate_motion_level_fast(self, current_frame):
        """Fast motion calculation for extraction decisions"""
        try:
            if len(self.frame_history) == 0:
                return 0.0

            current_gray = cv2.cvtColor(current_frame, cv2.COLOR_BGR2GRAY)
            last_gray = self.frame_history[-1]['gray_frame']

            # Resize for speed
            current_small = cv2.resize(current_gray, (160, 120))
            last_small = cv2.resize(last_gray, (160, 120))

            # Calculate difference
            diff = cv2.absdiff(current_small, last_small)
            motion_level = np.mean(diff) / 255.0

            return motion_level
        except:
            return 0.0

    def _quick_quality_assessment(self, frame):
        """Quick quality assessment for extraction decisions"""
        try:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            # Fast sharpness (Laplacian variance)
            laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
            sharpness_score = min(1.0, laplacian_var / 500.0)

            # Fast contrast
            contrast_score = min(1.0, gray.std() / 50.0)

            # Combined quality
            quality = (sharpness_score + contrast_score) / 2.0
            return quality
        except:
            return 0.0

    def _calculate_enhanced_basic_metrics(self, gray, frame):
        """Enhanced basic metrics for deepfake detection"""

        # Standard metrics
        sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        contrast = float(gray.std())
        brightness = float(gray.mean())

        # Enhanced metrics
        edges = cv2.Canny(gray, 50, 150)
        edge_density = float(len(edges.nonzero()[0]) / gray.size)

        # Texture analysis
        kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
        sharpened = cv2.filter2D(gray, -1, kernel)
        enhancement_response = float(np.mean(np.abs(sharpened - gray)))

        # Noise estimation
        noise_kernel = np.array([[1,-2,1],[-2,4,-2],[1,-2,1]])
        noise_response = cv2.filter2D(gray.astype(np.float32), -1, noise_kernel)
        noise_level = float(np.std(noise_response))

        return {
            'sharpness': sharpness,
            'contrast': contrast,
            'brightness': brightness,
            'edge_density': edge_density,
            'enhancement_response': enhancement_response,
            'noise_level': noise_level,
            'quality_score': min(1.0, (sharpness / 100 + contrast / 50) / 2)
        }

    def _analyze_faces_comprehensive(self, frame, gray):
        """Comprehensive face analysis for deepfake detection"""

        try:
            # Detect faces
            faces = self.face_cascade.detectMultiScale(gray, 1.1, 4, minSize=(50, 50))

            if len(faces) == 0:
                return {
                    'face_count': 0,
                    'faces_detected': False,
                    'face_quality': 0.0,
                    'face_consistency': 0.0,
                    'anomaly_flags': 0
                }

            face_metrics = []
            total_anomaly_flags = 0

            for (x, y, w, h) in faces:
                # Extract face region
                face_roi = frame[y:y+h, x:x+w]
                face_gray = gray[y:y+h, x:x+w]

                if face_roi.size == 0:
                    continue

                # Face quality metrics
                face_sharpness = cv2.Laplacian(face_gray, cv2.CV_64F).var()
                face_contrast = face_gray.std()

                # Face-specific deepfake indicators
                anomaly_flags = 0

                # 1. Boundary artifacts (common in deepfakes)
                boundary_score = self._analyze_face_boundaries(face_roi, x, y, w, h, frame.shape)
                if boundary_score > self.THRESHOLDS['boundary_artifact_threshold']:
                    anomaly_flags += 1

                # 2. Texture consistency
                texture_score = self._analyze_face_texture_consistency(face_roi)
                if texture_score > self.THRESHOLDS['texture_mismatch_threshold']:
                    anomaly_flags += 1

                # 3. Color space analysis
                color_anomaly = self._analyze_face_color_anomalies(face_roi)
                if color_anomaly > self.THRESHOLDS['color_space_anomaly']:
                    anomaly_flags += 1

                face_metrics.append({
                    'bbox': (x, y, w, h),
                    'sharpness': face_sharpness,
                    'contrast': face_contrast,
                    'boundary_score': boundary_score,
                    'texture_score': texture_score,
                    'color_anomaly': color_anomaly,
                    'anomaly_flags': anomaly_flags
                })

                total_anomaly_flags += anomaly_flags

            # Aggregate face metrics
            avg_face_quality = np.mean([f['sharpness'] + f['contrast'] for f in face_metrics]) / 2
            max_anomaly_flags = max([f['anomaly_flags'] for f in face_metrics])

            return {
                'face_count': len(faces),
                'faces_detected': True,
                'face_quality': float(avg_face_quality),
                'face_consistency': 1.0 - (total_anomaly_flags / (len(faces) * 3)),  # 3 tests per face
                'anomaly_flags': total_anomaly_flags,
                'individual_faces': face_metrics
            }

        except Exception as e:
            return {
                'face_count': 0,
                'faces_detected': False,
                'face_quality': 0.0,
                'face_consistency': 0.0,
                'anomaly_flags': 0,
                'error': str(e)
            }

    def _detect_deepfake_artifacts(self, frame, gray, hsv, lab):
        """Detect specific deepfake artifacts"""

        artifacts = {
            'compression_inconsistency': False,
            'gan_artifacts': False,
            'diffusion_smoothness': False,
            'frequency_anomalies': False,
            'color_bleeding': False,
            'total_flags': 0
        }

        try:
            # 1. Compression inconsistency detection
            compression_score = self._detect_compression_inconsistency(gray)
            if compression_score > self.THRESHOLDS['compression_inconsistency']:
                artifacts['compression_inconsistency'] = True
                artifacts['total_flags'] += 1

            # 2. GAN artifact detection
            gan_score = self._detect_gan_artifacts(gray)
            if gan_score > self.THRESHOLDS['gan_artifact_threshold']:
                artifacts['gan_artifacts'] = True
                artifacts['total_flags'] += 1

            # 3. Diffusion model over-smoothness
            smoothness_score = self._detect_diffusion_smoothness(gray)
            if smoothness_score > self.THRESHOLDS['diffusion_smoothness_max']:
                artifacts['diffusion_smoothness'] = True
                artifacts['total_flags'] += 1

            # 4. Frequency domain anomalies
            freq_anomaly_score = self._detect_frequency_anomalies_simple(gray)
            if freq_anomaly_score > self.THRESHOLDS['frequency_anomaly_min']:
                artifacts['frequency_anomalies'] = True
                artifacts['total_flags'] += 1

            # 5. Color bleeding (common in deepfakes)
            color_bleeding_score = self._detect_color_bleeding(hsv)
            if color_bleeding_score > 0.2:
                artifacts['color_bleeding'] = True
                artifacts['total_flags'] += 1

        except Exception as e:
            artifacts['error'] = str(e)

        return artifacts

    def _analyze_temporal_consistency(self, frame, gray, last_frame):
        """Analyze temporal consistency for deepfake detection"""

        if last_frame is None or len(self.frame_history) == 0:
            return {
                'temporal_consistency': 1.0,
                'motion_naturalness': 1.0,
                'lighting_consistency': 1.0,
                'inconsistency_flags': 0
            }

        try:
            last_gray = cv2.cvtColor(last_frame['frame'], cv2.COLOR_BGR2GRAY)

            inconsistency_flags = 0

            # 1. Frame-to-frame consistency
            frame_diff = cv2.absdiff(gray, last_gray)
            consistency_score = 1.0 - (np.mean(frame_diff) / 255.0)

            # 2. Motion analysis
            motion_score = self._analyze_motion_naturalness(gray, last_gray)

            # 3. Lighting consistency
            lighting_diff = abs(np.mean(gray) - np.mean(last_gray))
            lighting_consistency = 1.0 - min(1.0, lighting_diff / 50.0)

            # Flag inconsistencies
            if consistency_score < 0.7:  # Too much change
                inconsistency_flags += 1
            if motion_score > 0.4:  # Unnatural motion
                inconsistency_flags += 1
            if lighting_diff > self.THRESHOLDS['lighting_change_threshold']:
                inconsistency_flags += 1

            return {
                'temporal_consistency': float(consistency_score),
                'motion_naturalness': float(motion_score),
                'lighting_consistency': float(lighting_consistency),
                'inconsistency_flags': inconsistency_flags,
                'frame_difference': float(np.mean(frame_diff))
            }

        except Exception as e:
            return {
                'temporal_consistency': 1.0,
                'motion_naturalness': 1.0,
                'lighting_consistency': 1.0,
                'inconsistency_flags': 0,
                'error': str(e)
            }

    def _analyze_frequency_domain_detailed(self, gray):
        """Detailed frequency domain analysis for deepfake detection"""

        try:
            # Resize for consistent analysis
            if gray.shape[0] > 256 or gray.shape[1] > 256:
                gray = cv2.resize(gray, (256, 256))

            # DCT analysis
            dct = cv2.dct(gray.astype(np.float32))

            # Frequency energy distribution
            total_energy = np.sum(np.abs(dct))
            low_freq_energy = np.sum(np.abs(dct[:64, :64]))
            high_freq_energy = np.sum(np.abs(dct[128:, 128:]))

            freq_ratio = high_freq_energy / (total_energy + 1e-7)

            # FFT analysis for periodic artifacts
            fft_result = np.fft.fft2(gray)
            fft_magnitude = np.abs(np.fft.fftshift(fft_result))

            # Look for artificial periodicities
            periodicity_score = self._detect_artificial_periodicities(fft_magnitude)

            return {
                'frequency_ratio': float(freq_ratio),
                'periodicity_score': float(periodicity_score),
                'total_energy': float(total_energy),
                'artificial_patterns': periodicity_score > 0.3
            }

        except Exception as e:
            return {
                'frequency_ratio': 0.0,
                'periodicity_score': 0.0,
                'total_energy': 0.0,
                'artificial_patterns': False,
                'error': str(e)
            }

    def _analyze_texture_boundaries(self, frame, gray):
        """Analyze texture and boundary consistency"""

        try:
            # Edge detection
            edges = cv2.Canny(gray, 50, 150)
            edge_consistency = self._calculate_edge_consistency(edges)

            # Texture analysis
            texture_score = self._calculate_texture_regularity(gray)

            # Gradient analysis
            grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
            grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
            gradient_magnitude = np.sqrt(grad_x**2 + grad_y**2)
            gradient_consistency = 1.0 / (1.0 + np.std(gradient_magnitude))

            return {
                'edge_consistency': float(edge_consistency),
                'texture_regularity': float(texture_score),
                'gradient_consistency': float(gradient_consistency),
                'boundary_anomalies': edge_consistency > 0.9  # Too consistent = suspicious
            }

        except Exception as e:
            return {
                'edge_consistency': 0.0,
                'texture_regularity': 0.0,
                'gradient_consistency': 0.0,
                'boundary_anomalies': False,
                'error': str(e)
            }

    def _calculate_deepfake_suspicion_score(self, basic, face, artifacts, temporal, frequency, texture):
        """Calculate comprehensive deepfake suspicion score"""

        suspicion_factors = []

        # Face-based suspicion (most important for deepfakes)
        if face['faces_detected']:
            face_suspicion = (5 - face['face_consistency']) * 0.2  # 0-1 scale
            face_suspicion += face['anomaly_flags'] * 0.15
            suspicion_factors.append(('face_anomalies', min(1.0, face_suspicion)))

        # Artifact-based suspicion
        artifact_suspicion = artifacts['total_flags'] * 0.2
        suspicion_factors.append(('artifacts', min(1.0, artifact_suspicion)))

        # Temporal inconsistencies
        temporal_suspicion = temporal['inconsistency_flags'] * 0.25
        suspicion_factors.append(('temporal', min(1.0, temporal_suspicion)))

        # Frequency anomalies
        freq_suspicion = frequency.get('periodicity_score', 0) * 0.8
        suspicion_factors.append(('frequency', min(1.0, freq_suspicion)))

        # Quality inconsistencies
        quality_suspicion = 0.0
        if basic['quality_score'] > 0.95:  # Too perfect
            quality_suspicion += 0.3
        if basic['noise_level'] < 2.0:  # Too little noise
            quality_suspicion += 0.2
        suspicion_factors.append(('quality', min(1.0, quality_suspicion)))

        # Weighted average
        weights = {'face_anomalies': 0.35, 'artifacts': 0.25, 'temporal': 0.20, 'frequency': 0.15, 'quality': 0.05}

        total_suspicion = 0.0
        for factor_name, score in suspicion_factors:
            weight = weights.get(factor_name, 0.1)
            total_suspicion += score * weight

        # Determine suspicion level
        if total_suspicion >= 0.7:
            level = 'very_high'
        elif total_suspicion >= 0.5:
            level = 'high'
        elif total_suspicion >= 0.3:
            level = 'medium'
        else:
            level = 'low'

        return {
            'suspicion_score': float(total_suspicion),
            'suspicion_level': level,
            'contributing_factors': dict(suspicion_factors),
            'primary_concern': max(suspicion_factors, key=lambda x: x[1])[0] if suspicion_factors else 'none'
        }

    # HELPER METHODS FOR DETAILED ANALYSIS

    def _analyze_face_boundaries(self, face_roi, x, y, w, h, frame_shape):
        """Analyze face boundary artifacts (common in deepfakes)"""
        try:
            # Create boundary masks
            boundary_width = max(2, min(w, h) // 20)

            # Top boundary
            top_boundary = face_roi[:boundary_width, :]
            # Bottom boundary
            bottom_boundary = face_roi[-boundary_width:, :]
            # Left boundary
            left_boundary = face_roi[:, :boundary_width]
            # Right boundary
            right_boundary = face_roi[:, -boundary_width:]

            # Calculate boundary consistency
            boundaries = [top_boundary, bottom_boundary, left_boundary, right_boundary]
            boundary_scores = []

            for boundary in boundaries:
                if boundary.size > 0:
                    # Look for unnatural edges or artifacts
                    gray_boundary = cv2.cvtColor(boundary, cv2.COLOR_BGR2GRAY)
                    edge_strength = cv2.Laplacian(gray_boundary, cv2.CV_64F).var()
                    boundary_scores.append(edge_strength)

            # High edge strength at boundaries = suspicious
            avg_boundary_strength = np.mean(boundary_scores) if boundary_scores else 0
            return float(avg_boundary_strength)

        except Exception:
            return 0.0

    def _analyze_face_texture_consistency(self, face_roi):
        """Analyze texture consistency within face"""
        try:
            gray_face = cv2.cvtColor(face_roi, cv2.COLOR_BGR2GRAY)

            # Divide face into regions
            h, w = gray_face.shape
            regions = [
                gray_face[:h//2, :w//2],      # Top-left
                gray_face[:h//2, w//2:],      # Top-right
                gray_face[h//2:, :w//2],      # Bottom-left
                gray_face[h//2:, w//2:]       # Bottom-right
            ]

            # Calculate texture variance for each region
            texture_variances = []
            for region in regions:
                if region.size > 0:
                    # Use LBP-like texture measure
                    kernel = np.array([[1,1,1],[1,-8,1],[1,1,1]])
                    texture_response = cv2.filter2D(region.astype(np.float32), -1, kernel)
                    texture_var = np.var(texture_response)
                    texture_variances.append(texture_var)

            # Inconsistent texture = suspicious
            texture_inconsistency = np.std(texture_variances) if len(texture_variances) > 1 else 0
            return float(texture_inconsistency / 1000.0)  # Normalize

        except Exception:
            return 0.0

    def _analyze_face_color_anomalies(self, face_roi):
        """Analyze color space anomalies in face"""
        try:
            # Convert to different color spaces
            hsv_face = cv2.cvtColor(face_roi, cv2.COLOR_BGR2HSV)
            lab_face = cv2.cvtColor(face_roi, cv2.COLOR_BGR2LAB)

            # Analyze skin tone consistency
            h, s, v = cv2.split(hsv_face)
            l, a, b = cv2.split(lab_face)

            # Skin tones should be consistent
            hue_consistency = 1.0 - (np.std(h) / 180.0)
            saturation_consistency = 1.0 - (np.std(s) / 255.0)

            # Look for unnatural color shifts
            color_anomaly = 0.0
            if hue_consistency < 0.8:  # Too much hue variation
                color_anomaly += (0.8 - hue_consistency)
            if saturation_consistency < 0.7:  # Too much saturation variation
                color_anomaly += (0.7 - saturation_consistency)

            return float(color_anomaly * 50)  # Scale for threshold comparison

        except Exception:
            return 0.0

    def _detect_compression_inconsistency(self, gray):
        """Detect compression inconsistencies (common in deepfakes)"""
        try:
            # Analyze 8x8 blocks (JPEG compression blocks)
            h, w = gray.shape
            block_variances = []

            for i in range(0, h-8, 8):
                for j in range(0, w-8, 8):
                    block = gray[i:i+8, j:j+8]
                    block_var = np.var(block.astype(np.float32))
                    block_variances.append(block_var)

            if len(block_variances) > 1:
                # Inconsistent compression = suspicious
                compression_inconsistency = np.std(block_variances) / np.mean(block_variances)
                return float(compression_inconsistency)

            return 0.0
        except Exception:
            return 0.0

    def _detect_gan_artifacts(self, gray):
        """Detect GAN-specific artifacts"""
        try:
            # Look for checkerboard patterns (common GAN artifact)
            kernel_checkerboard = np.array([[1, -1, 1], [-1, 4, -1], [1, -1, 1]]) / 4.0
            checkerboard_response = cv2.filter2D(gray.astype(np.float32), -1, kernel_checkerboard)
            checkerboard_score = np.mean(np.abs(checkerboard_response))

            # Look for upsampling artifacts
            # Downsample and upsample to detect reconstruction artifacts
            small = cv2.resize(gray, (gray.shape[1]//4, gray.shape[0]//4))
            reconstructed = cv2.resize(small, (gray.shape[1], gray.shape[0]))

            reconstruction_diff = np.mean(np.abs(gray.astype(np.float32) - reconstructed.astype(np.float32)))

            gan_score = (checkerboard_score / 50.0) + (reconstruction_diff / 255.0)
            return float(min(1.0, gan_score))

        except Exception:
            return 0.0

    def _detect_diffusion_smoothness(self, gray):
        """Detect over-smoothness from diffusion models"""
        try:
            # Calculate local gradient magnitude
            grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
            grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
            gradient_magnitude = np.sqrt(grad_x**2 + grad_y**2)

            # Diffusion models often produce overly smooth results
            gradient_variance = np.var(gradient_magnitude)
            smoothness_score = 1.0 / (1.0 + gradient_variance / 100.0)

            return float(smoothness_score)

        except Exception:
            return 0.0

    def _detect_frequency_anomalies_simple(self, gray):
        """Simple frequency anomaly detection"""
        try:
            # DCT-based analysis
            dct_result = cv2.dct(gray.astype(np.float32))

            # Look for artificial frequency patterns
            # High frequencies should decrease naturally
            h, w = dct_result.shape

            # Compare different frequency bands
            low_freq = np.mean(np.abs(dct_result[:h//4, :w//4]))
            mid_freq = np.mean(np.abs(dct_result[h//4:h//2, w//4:w//2]))
            high_freq = np.mean(np.abs(dct_result[h//2:, w//2:]))

            # Natural images have decreasing frequency content
            if low_freq > 0:
                mid_ratio = mid_freq / low_freq
                high_ratio = high_freq / low_freq

                # Unnatural if high frequencies are too prominent or too suppressed
                if high_ratio > 0.5 or high_ratio < 0.05:
                    return float(abs(high_ratio - 0.2) * 2)

            return 0.0
        except Exception:
            return 0.0

    def _detect_color_bleeding(self, hsv):
        """Detect color bleeding artifacts"""
        try:
            h, s, v = cv2.split(hsv)

            # Look for unnatural color transitions
            # Calculate color gradients
            h_grad_x = cv2.Sobel(h, cv2.CV_64F, 1, 0, ksize=3)
            h_grad_y = cv2.Sobel(h, cv2.CV_64F, 0, 1, ksize=3)

            # Hue should change smoothly, not abruptly
            hue_gradient_magnitude = np.sqrt(h_grad_x**2 + h_grad_y**2)

            # Look for sharp hue transitions (color bleeding)
            sharp_transitions = np.sum(hue_gradient_magnitude > 30) / hue_gradient_magnitude.size

            return float(sharp_transitions)

        except Exception:
            return 0.0

    def _analyze_motion_naturalness(self, current_gray, last_gray):
        """Analyze naturalness of motion between frames"""
        try:
            # Resize for speed
            curr_small = cv2.resize(current_gray, (160, 120))
            last_small = cv2.resize(last_gray, (160, 120))

            # Calculate optical flow
            flow = cv2.calcOpticalFlowPyrLK(
                last_small, curr_small,
                cv2.goodFeaturesToTrack(last_small, maxCorners=100, qualityLevel=0.01, minDistance=10),
                None
            )[0]

            if flow is not None and len(flow) > 5:
                # Analyze flow consistency
                flow_magnitudes = np.linalg.norm(flow, axis=1)
                flow_consistency = 1.0 - (np.std(flow_magnitudes) / (np.mean(flow_magnitudes) + 1e-7))

                # Too consistent motion = suspicious
                if flow_consistency > 0.9:
                    return float(flow_consistency - 0.7)

            return 0.0
        except Exception:
            return 0.0

    def _detect_artificial_periodicities(self, fft_magnitude):
        """Detect artificial periodic patterns in frequency domain"""
        try:
            # Look for peaks that are too regular
            h, w = fft_magnitude.shape
            center_h, center_w = h//2, w//2

            # Sample radial frequencies
            max_radius = min(center_h, center_w) - 10
            radial_profile = []

            for r in range(5, max_radius, 2):
                # Create circular mask
                y, x = np.ogrid[:h, :w]
                mask = ((x - center_w)**2 + (y - center_h)**2 >= r**2) & \
                       ((x - center_w)**2 + (y - center_h)**2 < (r+2)**2)

                if np.any(mask):
                    radial_avg = np.mean(fft_magnitude[mask])
                    radial_profile.append(radial_avg)

            if len(radial_profile) > 3:
                # Look for artificial regularity
                profile_std = np.std(radial_profile)
                profile_mean = np.mean(radial_profile)

                if profile_mean > 0:
                    regularity = 1.0 - (profile_std / profile_mean)
                    return float(max(0, regularity - 0.7) * 5)  # Boost suspicious regularity

            return 0.0
        except Exception:
            return 0.0

    def _calculate_edge_consistency(self, edges):
        """Calculate consistency of edge patterns"""
        try:
            if np.sum(edges) == 0:
                return 0.0

            # Find edge pixels
            edge_pixels = np.where(edges > 0)

            if len(edge_pixels[0]) < 10:
                return 0.0

            # Calculate edge direction consistency
            # This is simplified - in practice you'd use Sobel gradients
            edge_orientations = []
            for i in range(0, len(edge_pixels[0]), 10):  # Sample every 10th edge pixel
                y, x = edge_pixels[0][i], edge_pixels[1][i]

                # Simple local orientation estimate
                if y > 1 and y < edges.shape[0]-2 and x > 1 and x < edges.shape[1]-2:
                    local_patch = edges[y-1:y+2, x-1:x+2]
                    if np.sum(local_patch) > 1:
                        # Simplified orientation
                        vertical_strength = np.sum(local_patch[:, 1])
                        horizontal_strength = np.sum(local_patch[1, :])

                        if horizontal_strength + vertical_strength > 0:
                            orientation = vertical_strength / (horizontal_strength + vertical_strength)
                            edge_orientations.append(orientation)

            if len(edge_orientations) > 5:
                orientation_consistency = 1.0 - (np.std(edge_orientations) / (np.mean(edge_orientations) + 1e-7))
                return float(max(0, orientation_consistency))

            return 0.0
        except Exception:
            return 0.0

    def _calculate_texture_regularity(self, gray):
        """Calculate texture regularity"""
        try:
            # Local Binary Pattern-like analysis
            kernel = np.array([[1, 1, 1], [1, -8, 1], [1, 1, 1]])
            texture_response = cv2.filter2D(gray.astype(np.float32), -1, kernel)

            # Calculate texture energy
            texture_energy = np.var(texture_response)

            # Normalize
            regularity_score = min(1.0, texture_energy / 1000.0)

            return float(regularity_score)

        except Exception:
            return 0.0

    def _update_deepfake_statistics(self, frame_analysis, deepfake_stats, extraction_reasons):
        """Update deepfake-specific statistics"""

        metadata = frame_analysis['metadata']

        # Count extraction reasons
        for reason in extraction_reasons:
            if reason in deepfake_stats['extraction_reasons']:
                deepfake_stats['extraction_reasons'][reason] += 1

        # Count face frames
        if metadata['face_metrics']['faces_detected']:
            deepfake_stats['face_frames'] += 1

        # Count high-quality frames
        if metadata['basic_metrics']['quality_score'] > 0.7:
            deepfake_stats['high_quality_frames'] += 1

        # Count suspicious frames
        suspicion_level = metadata['deepfake_suspicion']['suspicion_level']
        if suspicion_level in ['high', 'very_high']:
            deepfake_stats['suspicious_frames'] += 1

        # Count specific artifacts
        deepfake_stats['temporal_inconsistencies'] += metadata['temporal_metrics']['inconsistency_flags']
        deepfake_stats['boundary_artifacts'] += metadata['face_metrics']['anomaly_flags']
        deepfake_stats['total_deepfake_flags'] += metadata['total_flags']

    def _update_frame_history(self, frame_analysis):
        """Update frame history for temporal analysis"""

        frame = frame_analysis['frame']
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        self.frame_history.append({
            'frame_number': frame_analysis['metadata']['frame_number'],
            'gray_frame': gray,
            'suspicion_level': frame_analysis['metadata']['deepfake_suspicion']['suspicion_level'],
            'face_detected': frame_analysis['metadata']['face_metrics']['faces_detected'],
            'quality_score': frame_analysis['metadata']['basic_metrics']['quality_score']
        })

    def _organize_deepfake_results(self, extracted_frames, deepfake_stats, total_frames, duration, fps):
        """Organize results with deepfake-specific categorization"""

        # Categorize by deepfake suspicion level
        very_high_suspicion = [f for f in extracted_frames if f['metadata']['deepfake_suspicion']['suspicion_level'] == 'very_high']
        high_suspicion = [f for f in extracted_frames if f['metadata']['deepfake_suspicion']['suspicion_level'] == 'high']
        medium_suspicion = [f for f in extracted_frames if f['metadata']['deepfake_suspicion']['suspicion_level'] == 'medium']
        low_suspicion = [f for f in extracted_frames if f['metadata']['deepfake_suspicion']['suspicion_level'] == 'low']

        # Calculate comprehensive deepfake probability
        total_extracted = len(extracted_frames)
        total_flags = deepfake_stats['total_deepfake_flags']
        face_frame_ratio = deepfake_stats['face_frames'] / max(1, total_extracted)

        # Enhanced probability calculation
        base_probability = min(80, (total_flags / max(1, total_extracted * 3)) * 100)  # 3 potential flags per frame

        # Boost probability based on face content and suspicious frames
        face_boost = face_frame_ratio * 20  # Up to 20% boost for face content
        suspicion_boost = (deepfake_stats['suspicious_frames'] / max(1, total_extracted)) * 30  # Up to 30% boost for suspicious content

        deepfake_probability = min(95, base_probability + face_boost + suspicion_boost)

        # Determine verdict with updated thresholds
        if deepfake_probability >= 70:
            verdict = "VERY_HIGH"
            recommendation = "🚨 Strong deepfake indicators - likely synthetic content"
        elif deepfake_probability >= 50:
            verdict = "HIGH"
            recommendation = "⚠️ Multiple deepfake patterns detected - probably fake"
        elif deepfake_probability >= 30:
            verdict = "MEDIUM"
            recommendation = "🔍 Some deepfake indicators present - requires analysis"
        elif deepfake_probability >= 15:
            verdict = "LOW"
            recommendation = "✅ Minimal deepfake indicators - likely authentic"
        else:
            verdict = "VERY_LOW"
            recommendation = "✅ Strong authenticity indicators - appears genuine"

        return {
            # Suspicion level categories
            'very_high_suspicion_frames': very_high_suspicion,
            'high_suspicion_frames': high_suspicion,
            'medium_suspicion_frames': medium_suspicion,
            'low_suspicion_frames': low_suspicion,

            # All frames
            'all_frames': extracted_frames,

            # Deepfake analysis results
            'deepfake_analysis': {
                'total_video_frames': total_frames,
                'extracted_frames': total_extracted,
                'extraction_rate': total_extracted / total_frames,
                'duration_seconds': duration,
                'fps': fps,
                'deepfake_probability_percent': deepfake_probability,
                'verdict': verdict,
                'recommendation': recommendation,
                'face_content_ratio': face_frame_ratio,
                'detection_breakdown': {
                    'total_flags': total_flags,
                    'face_frames': deepfake_stats['face_frames'],
                    'suspicious_frames': deepfake_stats['suspicious_frames'],
                    'temporal_inconsistencies': deepfake_stats['temporal_inconsistencies'],
                    'boundary_artifacts': deepfake_stats['boundary_artifacts']
                },
                'extraction_strategy': deepfake_stats['extraction_reasons']
            },

            # Technical details
            'thresholds_used': self.THRESHOLDS,
            'extraction_strategy': self.EXTRACTION_STRATEGY
        }

    def _print_deepfake_analysis_summary(self, result):
        """Print comprehensive deepfake analysis summary"""

        analysis = result['deepfake_analysis']

        print(f"\n OPTIMAL DEEPFAKE ANALYSIS COMPLETE")
        print(f"=" * 60)

        print(f"📊 EXTRACTION SUMMARY:")
        print(f"   🎬 Total video frames: {analysis['total_video_frames']:,}")
        print(f"   📤 Extracted frames: {analysis['extracted_frames']} ({analysis['extraction_rate']:.1%})")
        print(f"   ⏱️  Duration: {analysis['duration_seconds']:.1f}s @ {analysis['fps']:.1f} FPS")
        print(f"   👤 Face content ratio: {analysis['face_content_ratio']:.1%}")

        print(f"\n🎭 DEEPFAKE DETECTION RESULTS:")
        print(f"   📊 Deepfake Probability: {analysis['deepfake_probability_percent']:.1f}%")
        print(f"   🎯 Verdict: {analysis['verdict']}")
        print(f"   💡 {analysis['recommendation']}")

        print(f"\n🔍 DETECTION BREAKDOWN:")
        breakdown = analysis['detection_breakdown']
        print(f"   🚩 Total flags: {breakdown['total_flags']}")
        print(f"   👤 Face frames: {breakdown['face_frames']}")
        print(f"   🚨 Suspicious frames: {breakdown['suspicious_frames']}")
        print(f"   ⏱️ Temporal inconsistencies: {breakdown['temporal_inconsistencies']}")
        print(f"   🎭 Boundary artifacts: {breakdown['boundary_artifacts']}")

        print(f"\n📈 FRAME DISTRIBUTION:")
        print(f"   🚨 Very High Suspicion: {len(result['very_high_suspicion_frames'])} frames")
        print(f"   ⚡ High Suspicion: {len(result['high_suspicion_frames'])} frames")
        print(f"   🟡 Medium Suspicion: {len(result['medium_suspicion_frames'])} frames")
        print(f"   🟢 Low Suspicion: {len(result['low_suspicion_frames'])} frames")

        print(f"\n🎯 EXTRACTION STRATEGY BREAKDOWN:")
        extraction_reasons = analysis['extraction_strategy']
        for reason, count in extraction_reasons.items():
            print(f"   📊 {reason.replace('_', ' ').title()}: {count} frames")



In [ ]:
import cv2
import numpy as np
import json
from collections import deque
import math
from sklearn.metrics.pairwise import cosine_similarity
import time
from datetime import datetime
import base64
import os

try:
    from deepface import DeepFace
    DEEPFACE_AVAILABLE = True
except Exception as e:
    print("⚠️ DeepFace not available. Reason:", type(e).__name__, "-", e)
    DEEPFACE_AVAILABLE = False


class ExpressionAwareFaceWeightDetector:
    def __init__(self):
        """
        Expression-aware face weight comparison detector that accounts for
        natural facial expressions while detecting deepfake inconsistencies
        """

        # Adjusted thresholds for expression-aware detection
        self.THRESHOLDS = {
            'identity_core_similarity': 0.70,        # Core identity (expression-invariant)
            'expression_variance_limit': 0.25,       # Max natural expression variance
            'identity_break_threshold': 0.55,        # Clear identity switch (lowered)
            'sudden_identity_change': 0.30,          # Very sudden identity change
            'temporal_consistency_window': 5,        # Frames to check for consistency
            'min_face_confidence': 0.6,
            'min_face_size': 48,
            'expression_smoothness_threshold': 0.40, # Expression changes should be smooth
            'unnatural_stability_threshold': 0.98,   # Too stable = suspicious
        }

        # Models for face analysis
        self.models = {
            'detector': 'opencv',
            'embedder': 'Facenet',  # 128-dimensional face weights
        }

        # Storage for analysis
        self.frame_classifications = []
        self.face_weights_history = []
        self.weight_comparisons = []
        self.expression_analysis = []
        self.temporal_windows = []

        print("TEMPORAL ANALYSIS INITIALIZED")


    def analyze_expression_aware_face_patterns(self, extracted_frames_result, max_frames=50):
        """
        Analyze face patterns with expression awareness
        """

        if not DEEPFACE_AVAILABLE:
            print("DEEPFACE NOT AVAILABLEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE")
            return self._create_opencv_fallback()

        frames_to_analyze = self._select_frames_for_analysis(extracted_frames_result, max_frames)

        print(f"\n🎭 EXPRESSION-AWARE FACE ANALYSIS")
        print(f"   📊 Analyzing {len(frames_to_analyze)} frames")
        print(f"   🎯 Distinguishing expressions from identity changes")
        print("=" * 60)

        start_time = time.time()

        # Extract face weights with expression context
        self._extract_face_weights_with_expression_context(frames_to_analyze)

        # Analyze temporal consistency with expression tolerance
        self._analyze_temporal_consistency_with_expressions()

        # Classify frames with expression awareness
        self._classify_frames_with_expression_awareness()

        total_time = time.time() - start_time

        # Generate results
        results = self._generate_expression_aware_results(total_time)

        # Generate HTML report
        html_report_path = self._generate_html_report(results)
        results['html_report_path'] = html_report_path

        self._print_expression_aware_analysis(results)

        return results

    def _select_frames_for_analysis(self, extracted_result, max_frames):
        """Select frames with better temporal distribution"""

        all_frames = []
        categories = ['very_high_suspicion_frames', 'high_suspicion_frames',
                     'medium_suspicion_frames', 'low_suspicion_frames']

        for category in categories:
            frames = extracted_result.get(category, [])
            all_frames.extend(frames)

        # Sort by frame number for temporal analysis
        all_frames.sort(key=lambda x: x['metadata']['frame_number'])

        # Select frames with better temporal distribution
        if len(all_frames) <= max_frames:
            return all_frames

        # Take every nth frame to maintain temporal coverage
        step = len(all_frames) // max_frames
        selected = all_frames[::step][:max_frames]

        # Always include the first and last few frames
        if len(all_frames) > 10:
            selected[:3] = all_frames[:3]  # First 3 frames
            selected[-2:] = all_frames[-2:]  # Last 2 frames

        print(f"   ✅ Selected {len(selected)} frames with temporal distribution")
        return selected

    def _extract_face_weights_with_expression_context(self, frames_to_analyze):
        """Extract face weights with additional expression context"""

        print(f"\n⚖️ EXTRACTING FACE WEIGHTS WITH EXPRESSION CONTEXT:")

        for i, frame_data in enumerate(frames_to_analyze):
            frame = frame_data['frame']
            frame_meta = frame_data['metadata']
            frame_number = frame_meta['frame_number']
            timestamp = frame_meta['timestamp']

            try:
                print(f"   📊 Processing Frame {frame_number} ({i+1}/{len(frames_to_analyze)}) @ {timestamp:.2f}s")

                # Extract faces with alignment for consistent analysis
                faces = DeepFace.extract_faces(
                    img_path=frame,
                    detector_backend=self.models['detector'],
                    enforce_detection=False,
                    align=True,
                    expand_percentage=15  # Slightly more context
                )

                if not faces:
                    self._add_failed_frame(frame_number, timestamp, "No face detected")
                    continue

                # Get primary face
                primary_face = faces[0]
                face_img = primary_face['face']
                face_region = primary_face['facial_area']

                # Extract core identity embedding (expression-invariant features)
                embedding_result = DeepFace.represent(
                    img_path=(face_img * 255).astype(np.uint8),
                    model_name=self.models['embedder'],
                    detector_backend=self.models['detector'],
                    enforce_detection=False
                )

                if not embedding_result or not embedding_result[0].get('embedding'):
                    self._add_failed_frame(frame_number, timestamp, "Failed to extract embeddings")
                    continue

                face_weights = embedding_result[0]['embedding']

                # Calculate additional metrics for expression analysis
                face_gray = cv2.cvtColor((face_img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)

                # Face quality metrics
                sharpness = cv2.Laplacian(face_gray, cv2.CV_64F).var()
                face_size = face_region['w'] * face_region['h']

                # Expression-related metrics
                face_contrast = face_gray.std()

                # Estimate pose variation (simple approximation)
                face_center_x = (face_region['x'] + face_region['w'] / 2) / frame.shape[1]
                face_center_y = (face_region['y'] + face_region['h'] / 2) / frame.shape[0]

                # Store comprehensive face data
                face_data = {
                    'frame_number': frame_number,
                    'timestamp': timestamp,
                    'has_face': True,
                    'weights': face_weights,
                    'weights_norm': np.linalg.norm(face_weights),
                    'face_region': face_region,
                    'face_size': face_size,
                    'face_center': (face_center_x, face_center_y),
                    'sharpness': float(sharpness),
                    'contrast': float(face_contrast),
                    'confidence': primary_face.get('confidence', 0.8),
                    'original_suspicion': frame_data.get('suspicion_level', 'unknown'),
                    'face_image_b64': self._encode_face_to_base64(face_img)  # For HTML report
                }

                self.face_weights_history.append(face_data)

                print(f"      ✅ Weights: {len(face_weights)}D, norm={np.linalg.norm(face_weights):.3f}, size={face_size}")

            except Exception as e:
                self._add_failed_frame(frame_number, timestamp, str(e))
                print(f"      ❌ Error processing frame {frame_number}: {e}")

        valid_weights = len([f for f in self.face_weights_history if f['weights'] is not None])
        print(f"\n   ✅ Successfully processed {valid_weights} frames")

    def _encode_face_to_base64(self, face_img):
        """Encode face image to base64 for HTML report"""
        try:
            # Convert to uint8 and encode as PNG
            face_uint8 = (face_img * 255).astype(np.uint8)
            face_bgr = cv2.cvtColor(face_uint8, cv2.COLOR_RGB2BGR)

            # Encode to PNG
            _, buffer = cv2.imencode('.png', face_bgr)
            img_b64 = base64.b64encode(buffer).decode('utf-8')
            return f"data:image/png;base64,{img_b64}"
        except:
            return None

    def _add_failed_frame(self, frame_number, timestamp, error):
        """Add failed frame to history"""
        self.face_weights_history.append({
            'frame_number': frame_number,
            'timestamp': timestamp,
            'has_face': False,
            'weights': None,
            'extraction_error': error,
            'face_image_b64': None
        })

    def _analyze_temporal_consistency_with_expressions(self):
        """Analyze temporal consistency accounting for natural expressions"""

        print(f"\n🔍 ANALYZING TEMPORAL CONSISTENCY WITH EXPRESSION TOLERANCE:")

        valid_frames = [f for f in self.face_weights_history if f['weights'] is not None]

        if len(valid_frames) < 3:
            print("   ❌ Insufficient frames for temporal analysis")
            return

        # Calculate baseline identity from multiple reference frames
        reference_count = min(5, len(valid_frames) // 3)  # Use first 1/3 as reference
        reference_frames = valid_frames[:reference_count]
        reference_weights = np.mean([f['weights'] for f in reference_frames], axis=0)

        print(f"   📊 Using {reference_count} frames for identity baseline")

        # Analyze each frame in temporal windows
        window_size = self.THRESHOLDS['temporal_consistency_window']

        for i in range(len(valid_frames)):
            current_frame = valid_frames[i]
            frame_number = current_frame['frame_number']
            current_weights = current_frame['weights']

            # Create temporal window around current frame
            window_start = max(0, i - window_size // 2)
            window_end = min(len(valid_frames), i + window_size // 2 + 1)
            window_frames = valid_frames[window_start:window_end]

            # Analyze consistency within temporal window
            window_similarities = []
            for window_frame in window_frames:
                if window_frame['frame_number'] != frame_number:
                    similarity = cosine_similarity([current_weights], [window_frame['weights']])[0][0]
                    window_similarities.append(similarity)

            # Calculate identity consistency scores
            identity_vs_reference = cosine_similarity([current_weights], [reference_weights])[0][0]
            window_consistency = np.mean(window_similarities) if window_similarities else 0.0
            window_stability = np.std(window_similarities) if len(window_similarities) > 1 else 0.0

            # Calculate previous and next frame similarities
            prev_similarity = None
            next_similarity = None

            if i > 0:
                prev_weights = valid_frames[i-1]['weights']
                prev_similarity = cosine_similarity([current_weights], [prev_weights])[0][0]

                # Calculate time gap for context
                time_gap = current_frame['timestamp'] - valid_frames[i-1]['timestamp']

                # Store frame comparison
                comparison_data = {
                    'frame_pair': f"{valid_frames[i-1]['frame_number']}->{frame_number}",
                    'previous_frame': valid_frames[i-1]['frame_number'],
                    'current_frame': frame_number,
                    'similarity': prev_similarity,
                    'time_gap': time_gap,
                    'expected_similarity_range': self._calculate_expected_similarity(time_gap),
                    'is_natural_change': self._is_natural_expression_change(prev_similarity, time_gap),
                    'position_change': self._calculate_position_change(valid_frames[i-1], current_frame),
                    'size_change': abs(current_frame['face_size'] - valid_frames[i-1]['face_size']) / max(current_frame['face_size'], 1)
                }

                self.weight_comparisons.append(comparison_data)

            if i < len(valid_frames) - 1:
                next_weights = valid_frames[i+1]['weights']
                next_similarity = cosine_similarity([current_weights], [next_weights])[0][0]

            # Store temporal analysis
            temporal_analysis = {
                'frame_number': frame_number,
                'timestamp': current_frame['timestamp'],
                'identity_vs_reference': identity_vs_reference,
                'window_consistency': window_consistency,
                'window_stability': window_stability,
                'prev_similarity': prev_similarity,
                'next_similarity': next_similarity,
                'temporal_window_size': len(window_frames),
                'is_identity_consistent': identity_vs_reference >= self.THRESHOLDS['identity_core_similarity'],
                'is_temporally_smooth': window_stability < 0.1,  # Low variance = smooth
                'is_unnaturally_stable': window_consistency > self.THRESHOLDS['unnatural_stability_threshold']
            }

            self.temporal_windows.append(temporal_analysis)

            # Print analysis for key frames
            if identity_vs_reference < 0.6 or (prev_similarity and prev_similarity < 0.4):
                status = "🚨 SUSPICIOUS" if identity_vs_reference < 0.6 else "⚠️ EXPRESSION"
                print(f"   Frame {frame_number}: {status}")
                print(f"      Identity vs Reference: {identity_vs_reference:.3f}")
                print(f"      Window Consistency: {window_consistency:.3f}")
                if prev_similarity:
                    print(f"      Previous Frame Similarity: {prev_similarity:.3f}")

    def _calculate_expected_similarity(self, time_gap):
        """Calculate expected similarity range based on time gap"""
        # Natural expressions change more over longer time periods
        if time_gap < 0.1:  # Very close frames
            return (0.85, 0.99)
        elif time_gap < 0.5:  # Close frames
            return (0.75, 0.95)
        elif time_gap < 1.0:  # Medium gap
            return (0.65, 0.90)
        else:  # Longer gap
            return (0.55, 0.85)

    def _is_natural_expression_change(self, similarity, time_gap):
        """Determine if similarity change is natural given time gap"""
        expected_min, expected_max = self._calculate_expected_similarity(time_gap)

        # If similarity is within expected range, it's natural
        if expected_min <= similarity <= expected_max:
            return True

        # If similarity is too high for the time gap, it might be unnatural
        if similarity > expected_max and time_gap > 0.5:
            return False  # Too stable over time

        # If similarity is too low for a short time gap, it's suspicious
        if similarity < expected_min and time_gap < 0.3:
            return False  # Too abrupt for short time

        return True

    def _calculate_position_change(self, frame1, frame2):
        """Calculate position change between frames"""
        center1 = frame1['face_center']
        center2 = frame2['face_center']

        return math.sqrt((center2[0] - center1[0])**2 + (center2[1] - center1[1])**2)

    def _classify_frames_with_expression_awareness(self):
        """Classify frames accounting for natural expressions"""

        print(f"\n🏷️ CLASSIFYING FRAMES WITH EXPRESSION AWARENESS:")

        for temporal_data in self.temporal_windows:
            frame_number = temporal_data['frame_number']

            # Find corresponding frame data
            frame_data = next(f for f in self.face_weights_history if f['frame_number'] == frame_number)

            # Classification scoring with expression tolerance
            suspicion_score = 0
            classification_reasons = []

            # 1. Identity consistency check (most important)
            identity_similarity = temporal_data['identity_vs_reference']
            if identity_similarity < self.THRESHOLDS['sudden_identity_change']:
                classification_reasons.append(f"🚨 SUDDEN IDENTITY CHANGE: Reference similarity = {identity_similarity:.3f} (< {self.THRESHOLDS['sudden_identity_change']})")
                suspicion_score += 50
            elif identity_similarity < self.THRESHOLDS['identity_break_threshold']:
                classification_reasons.append(f"⚠️ IDENTITY INCONSISTENCY: Reference similarity = {identity_similarity:.3f} (< {self.THRESHOLDS['identity_break_threshold']})")
                suspicion_score += 35
            elif identity_similarity < self.THRESHOLDS['identity_core_similarity']:
                classification_reasons.append(f"🔍 WEAK IDENTITY: Reference similarity = {identity_similarity:.3f} (< {self.THRESHOLDS['identity_core_similarity']})")
                suspicion_score += 20
            else:
                classification_reasons.append(f"✅ STRONG IDENTITY: Reference similarity = {identity_similarity:.3f}")

            # 2. Temporal smoothness analysis
            if temporal_data['is_unnaturally_stable']:
                classification_reasons.append(f"⚠️ UNNATURALLY STABLE: Window consistency = {temporal_data['window_consistency']:.3f} (too perfect)")
                suspicion_score += 25
            elif not temporal_data['is_temporally_smooth']:
                classification_reasons.append(f"🔍 TEMPORAL INCONSISTENCY: Window stability = {temporal_data['window_stability']:.3f}")
                suspicion_score += 15
            else:
                classification_reasons.append(f"✅ SMOOTH TEMPORAL CHANGE: Natural expression variation")

            # 3. Frame-to-frame analysis with expression context
            prev_sim = temporal_data['prev_similarity']
            next_sim = temporal_data['next_similarity']

            # Find corresponding comparison data
            prev_comparison = next((c for c in self.weight_comparisons if c['current_frame'] == frame_number), None)

            if prev_comparison:
                if not prev_comparison['is_natural_change']:
                    if prev_comparison['time_gap'] < 0.3 and prev_sim < 0.4:
                        classification_reasons.append(f"🚨 ABRUPT IDENTITY CHANGE: Previous frame similarity = {prev_sim:.3f} in {prev_comparison['time_gap']:.2f}s")
                        suspicion_score += 40
                    elif prev_sim > 0.98 and prev_comparison['time_gap'] > 0.5:
                        classification_reasons.append(f"⚠️ UNNATURALLY STABLE: Too similar over {prev_comparison['time_gap']:.2f}s")
                        suspicion_score += 20
                    else:
                        classification_reasons.append(f"🔍 UNUSUAL CHANGE PATTERN: Prev similarity = {prev_sim:.3f}")
                        suspicion_score += 10
                else:
                    classification_reasons.append(f"✅ NATURAL EXPRESSION CHANGE: Previous similarity = {prev_sim:.3f}")

            # 4. Quality and context factors
            confidence = frame_data['confidence']
            sharpness = frame_data['sharpness']
            original_suspicion = frame_data['original_suspicion']

            if confidence < 0.7:
                classification_reasons.append(f"⚠️ LOW FACE CONFIDENCE: {confidence:.3f}")
                suspicion_score += 10

            if sharpness < 100:
                classification_reasons.append(f"🔍 LOW QUALITY: Sharpness = {sharpness:.1f}")
                suspicion_score += 5

            if original_suspicion in ['very_high', 'high']:
                classification_reasons.append(f"🔴 PRE-CLASSIFIED SUSPICIOUS: {original_suspicion}")
                suspicion_score += 10

            # Final classification with expression-aware thresholds
            if suspicion_score >= 80:
                final_classification = "🚨 LIKELY FAKE"
                confidence_level = "HIGH"
            elif suspicion_score >= 60:
                final_classification = "⚠️ SUSPICIOUS"
                confidence_level = "HIGH"
            elif suspicion_score >= 40:
                final_classification = "🔍 QUESTIONABLE"
                confidence_level = "MEDIUM"
            elif suspicion_score >= 20:
                final_classification = "🟡 MINOR CONCERNS"
                confidence_level = "MEDIUM"
            else:
                final_classification = "✅ APPEARS REAL"
                confidence_level = "HIGH"

            frame_classification = {
                'frame_number': frame_number,
                'timestamp': frame_data['timestamp'],
                'classification': final_classification,
                'confidence_level': confidence_level,
                'suspicion_score': suspicion_score,
                'identity_similarity': identity_similarity,
                'temporal_consistency': temporal_data['window_consistency'],
                'reasons': classification_reasons,
                'face_confidence': confidence,
                'sharpness': sharpness,
                'face_image_b64': frame_data.get('face_image_b64'),
                'original_suspicion': original_suspicion
            }

            self.frame_classifications.append(frame_classification)

            # Print classification for problematic frames
            if suspicion_score >= 40:
                print(f"\n   Frame {frame_number} @ {frame_data['timestamp']:.2f}s: {final_classification} (Score: {suspicion_score})")
                for reason in classification_reasons[:2]:  # Show top 2 reasons
                    print(f"      {reason}")

    def _generate_expression_aware_results(self, processing_time):
        """Generate comprehensive results"""

        # Count classifications
        likely_fake = len([f for f in self.frame_classifications if "LIKELY FAKE" in f['classification']])
        suspicious = len([f for f in self.frame_classifications if "SUSPICIOUS" in f['classification']])
        questionable = len([f for f in self.frame_classifications if "QUESTIONABLE" in f['classification']])
        minor_concerns = len([f for f in self.frame_classifications if "MINOR CONCERNS" in f['classification']])
        appears_real = len([f for f in self.frame_classifications if "APPEARS REAL" in f['classification']])

        total_frames = len(self.frame_classifications)

        # Calculate overall assessment with expression awareness
        problematic_ratio = (likely_fake + 0.7 * suspicious) / max(total_frames, 1)

        # Calculate different concern levels
        # Calculate concern levels (as proportions 0-1)
        severe_concerns = (likely_fake + 0.8 * suspicious) / max(total_frames, 1)
        moderate_concerns = (likely_fake + suspicious + 0.6 * questionable) / max(total_frames, 1)
        any_concerns = (total_frames - appears_real) / max(total_frames, 1)

        # Multi-tier logic with corrected thresholds
        if severe_concerns > 0.10:  # 25%+ severe issues (likely_fake + weighted suspicious)
            overall_assessment = "🚨 LIKELY DEEPFAKE"
            deepfake_probability = min(85 + int(severe_concerns * 15), 95)  # 85-95% based on severity
        elif moderate_concerns > 0.15:  # 40%+ moderate+ issues
            overall_assessment = "⚠️ SUSPICIOUS CONTENT"
            deepfake_probability = min(60 + int(moderate_concerns * 25), 80)  # 60-80% based on concerns
        elif severe_concerns > 0.20 or appears_real < 0.30:  # 60%+ have concerns OR <30% appears real
            overall_assessment = "🔍 NEEDS INVESTIGATION"
            deepfake_probability = min(35 + int(any_concerns * 30), 65)  # 35-65% based on concerns
        else:
            overall_assessment = "✅ LIKELY AUTHENTIC"
            deepfake_probability = max(15 - int((1 - any_concerns) * 10), 5)  # 5-15% low confidence

        # Temporal analysis statistics
        identity_breaks = len([t for t in self.temporal_windows if t['identity_vs_reference'] < 0.6])
        unnatural_stability = len([t for t in self.temporal_windows if t['is_unnaturally_stable']])
        natural_changes = len([c for c in self.weight_comparisons if c['is_natural_change']])

        results = {
            'overall_assessment': {
                'verdict': overall_assessment,
                'deepfake_probability': deepfake_probability,
                'confidence': 'HIGH' if total_frames > 15 else 'MEDIUM'
            },

            'frame_statistics': {
                'total_frames_analyzed': total_frames,
                'likely_fake_frames': likely_fake,
                'suspicious_frames': suspicious,
                'questionable_frames': questionable,
                'minor_concerns_frames': minor_concerns,
                'appears_real_frames': appears_real,
                'problematic_percentage': problematic_ratio * 100
            },

            'temporal_analysis': {
                'identity_breaks': identity_breaks,
                'unnatural_stability_count': unnatural_stability,
                'natural_expression_changes': natural_changes,
                'total_frame_comparisons': len(self.weight_comparisons),
                'temporal_windows_analyzed': len(self.temporal_windows)
            },

            'detailed_classifications': self.frame_classifications,
            'temporal_data': self.temporal_windows,
            'frame_comparisons': self.weight_comparisons,

            'processing_stats': {
                'processing_time': processing_time,
                'frames_per_second': total_frames / max(processing_time, 0.001)
            }
        }

        return results

    def _generate_html_report(self, results):
        """Generate detailed HTML report with frame images"""

        report_filename = f"face_analysis_report_{int(time.time())}.html"

        # Get problematic frames for detailed display
        problematic_frames = [f for f in self.frame_classifications if f['suspicion_score'] >= 30]
        problematic_frames.sort(key=lambda x: x['suspicion_score'], reverse=True)

        html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Face Weight Analysis Report</title>
    <meta charset="UTF-8">
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; background-color: #f5f5f5; }}
        .header {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 10px; margin-bottom: 20px; }}
        .summary {{ background: white; padding: 20px; border-radius: 10px; margin-bottom: 20px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }}
        .frame-grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(300px, 1fr)); gap: 20px; }}
        .frame-card {{ background: white; border-radius: 10px; padding: 15px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }}
        .frame-card.fake {{ border-left: 5px solid #ff4444; }}
        .frame-card.suspicious {{ border-left: 5px solid #ff8800; }}
        .frame-card.questionable {{ border-left: 5px solid #ffcc00; }}
        .frame-card.real {{ border-left: 5px solid #44ff44; }}
        .face-image {{ width: 100px; height: 100px; border-radius: 50%; object-fit: cover; float: right; margin-left: 15px; }}
        .classification {{ font-weight: bold; font-size: 18px; margin-bottom: 10px; }}
        .timestamp {{ color: #666; font-size: 14px; }}
        .reasons {{ margin-top: 10px; }}
        .reason {{ margin: 5px 0; padding: 5px; background: #f8f9fa; border-radius: 5px; font-size: 14px; }}
        .stats {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin: 20px 0; }}
        .stat-card {{ background: white; padding: 15px; border-radius: 8px; text-align: center; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }}
        .stat-value {{ font-size: 24px; font-weight: bold; color: #667eea; }}
        .stat-label {{ color: #666; margin-top: 5px; }}
        .verdict {{ font-size: 24px; font-weight: bold; text-align: center; padding: 20px; margin: 20px 0; border-radius: 10px; }}
        .verdict.authentic {{ background: #d4edda; color: #155724; }}
        .verdict.suspicious {{ background: #fff3cd; color: #856404; }}
        .verdict.fake {{ background: #f8d7da; color: #721c24; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>🎭 DeepFake Analysis Report</h1>
        <p>Generated on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
    </div>

    <div class="summary">
        <div class="verdict {'fake' if 'DEEPFAKE' in results['overall_assessment']['verdict'] else 'suspicious' if 'SUSPICIOUS' in results['overall_assessment']['verdict'] else 'authentic'}">
            {results['overall_assessment']['verdict']}

        </div>

        <div class="stats">
            <div class="stat-card">
                <div class="stat-value">{results['frame_statistics']['total_frames_analyzed']}</div>
                <div class="stat-label">Total Frames</div>
            </div>
            <div class="stat-card">
                <div class="stat-value">{results['frame_statistics']['likely_fake_frames']}</div>
                <div class="stat-label">🚨 Likely Fake</div>
            </div>
            <div class="stat-card">
                <div class="stat-value">{results['frame_statistics']['suspicious_frames']}</div>
                <div class="stat-label">⚠️ Suspicious</div>
            </div>
            <div class="stat-card">
                <div class="stat-value">{results['frame_statistics']['appears_real_frames']}</div>
                <div class="stat-label">✅ Appears Real</div>
            </div>
            <div class="stat-card">
                <div class="stat-value">{results['temporal_analysis']['identity_breaks']}</div>
                <div class="stat-label">Identity Breaks</div>
            </div>
            <div class="stat-card">
                <div class="stat-value">{results['temporal_analysis']['natural_expression_changes']}</div>
                <div class="stat-label">Natural Changes</div>
            </div>
        </div>
    </div>

    <h2>📊 Detailed Frame Analysis</h2>
    <p>Showing frames with suspicion score ≥ 30. Frames are ordered by suspicion level.</p>

    <div class="frame-grid">"""

        # Add frame cards
        for frame in problematic_frames[:20]:  # Show top 20 problematic frames
            classification_class = "fake" if "FAKE" in frame['classification'] else \
                                 "suspicious" if "SUSPICIOUS" in frame['classification'] else \
                                 "questionable" if "QUESTIONABLE" in frame['classification'] else "real"

            face_img_html = ""
            if frame.get('face_image_b64'):
                face_img_html = f'<img src="{frame["face_image_b64"]}" alt="Face" class="face-image">'

            reasons_html = ""
            for reason in frame['reasons'][:3]:  # Show top 3 reasons
                reasons_html += f'<div class="reason">{reason}</div>'

            html_content += f"""
        <div class="frame-card {classification_class}">
            {face_img_html}
            <div class="classification">{frame['classification']}</div>
            <div class="timestamp">
                Frame {frame['frame_number']} @ {frame['timestamp']:.2f}s
                <br>Score: {frame['suspicion_score']} | Identity: {frame['identity_similarity']:.3f}
            </div>
            <div class="reasons">
                <strong>Key Issues:</strong>
                {reasons_html}
            </div>
        </div>"""

        # Add some authentic frames for comparison
        authentic_frames = [f for f in self.frame_classifications if "REAL" in f['classification']][:5]
        if authentic_frames:
            html_content += """
    </div>

    <h2>✅ Sample Authentic Frames (For Comparison)</h2>
    <div class="frame-grid">"""

            for frame in authentic_frames:
                face_img_html = ""
                if frame.get('face_image_b64'):
                    face_img_html = f'<img src="{frame["face_image_b64"]}" alt="Face" class="face-image">'

                html_content += f"""
        <div class="frame-card real">
            {face_img_html}
            <div class="classification">{frame['classification']}</div>
            <div class="timestamp">
                Frame {frame['frame_number']} @ {frame['timestamp']:.2f}s
                <br>Score: {frame['suspicion_score']} | Identity: {frame['identity_similarity']:.3f}
            </div>
        </div>"""

        html_content += """
    </div>

    <div class="summary" style="margin-top: 30px;">
        <h3>🔍 Analysis Methodology</h3>
        <p><strong>Expression-Aware Detection:</strong> This analysis distinguishes between natural facial expressions and artificial face swaps by:</p>
        <ul>
            <li>🎯 <strong>Identity Core Analysis:</strong> Focuses on expression-invariant facial features</li>
            <li>⏱️ <strong>Temporal Context:</strong> Considers time gaps between frames for natural expression changes</li>
            <li>🎭 <strong>Expression Tolerance:</strong> Allows for natural emotional variations while detecting abrupt identity changes</li>
            <li>📊 <strong>Window-Based Consistency:</strong> Analyzes facial consistency across multiple frame windows</li>
        </ul>

        <h4>🎯 Classification Criteria:</h4>
        <ul>
            <li>🚨 <strong>Likely Fake :</strong> Sudden identity changes, unnatural temporal patterns</li>
            <li>⚠️ <strong>Suspicious :</strong> Multiple concerning indicators</li>
            <li>🔍 <strong>Questionable :</strong> Some inconsistencies that need investigation</li>
            <li>✅ <strong>Appears Real :</strong> Natural expression patterns, consistent identity</li>
        </ul>
    </div>

</body>
</html>"""

        # Save HTML report
        try:
            with open(report_filename, 'w', encoding='utf-8') as f:
                f.write(html_content)
            print(f"   📄 HTML report saved: {report_filename}")
            return os.path.abspath(report_filename)
        except Exception as e:
            print(f"   ❌ Failed to save HTML report: {e}")
            return None

    def _print_expression_aware_analysis(self, results):
        """Print comprehensive analysis results"""

        overall = results['overall_assessment']
        stats = results['frame_statistics']
        temporal = results['temporal_analysis']
        processing = results['processing_stats']

        print(f"\n" + "="*80)
        print(f"📋 EXPRESSION-AWARE FACE ANALYSIS RESULTS")
        print(f"="*80)

        print(f"\n🎯 OVERALL ASSESSMENT:")
        print(f"   Verdict: {overall['verdict']}")
        print(f"   Deepfake Probability: {overall['deepfake_probability']:.1f}%")
        print(f"   Analysis Confidence: {overall['confidence']}")

        print(f"\n📊 FRAME CLASSIFICATION BREAKDOWN:")
        print(f"   Total Frames: {stats['total_frames_analyzed']}")
        print(f"   🚨 Likely Fake: {stats['likely_fake_frames']} ({(stats['likely_fake_frames']/max(stats['total_frames_analyzed'],1))*100:.1f}%)")
        print(f"   ⚠️ Suspicious: {stats['suspicious_frames']} ({(stats['suspicious_frames']/max(stats['total_frames_analyzed'],1))*100:.1f}%)")
        print(f"   🔍 Questionable: {stats['questionable_frames']} ({(stats['questionable_frames']/max(stats['total_frames_analyzed'],1))*100:.1f}%)")
        print(f"   🟡 Minor Concerns: {stats['minor_concerns_frames']} ({(stats['minor_concerns_frames']/max(stats['total_frames_analyzed'],1))*100:.1f}%)")
        print(f"   ✅ Appears Real: {stats['appears_real_frames']} ({(stats['appears_real_frames']/max(stats['total_frames_analyzed'],1))*100:.1f}%)")

        print(f"\n🎭 TEMPORAL CONSISTENCY ANALYSIS:")
        print(f"   Identity Breaks Detected: {temporal['identity_breaks']}")
        print(f"   Unnaturally Stable Periods: {temporal['unnatural_stability_count']}")
        print(f"   Natural Expression Changes: {temporal['natural_expression_changes']}")
        print(f"   Frame-to-Frame Comparisons: {temporal['total_frame_comparisons']}")

        print(f"\n⚡ PROCESSING PERFORMANCE:")
        print(f"   Processing Time: {processing['processing_time']:.1f}s")
        print(f"   Speed: {processing['frames_per_second']:.1f} frames/second")

        # Show most problematic frames
        problematic = [f for f in self.frame_classifications if f['suspicion_score'] >= 50]
        problematic.sort(key=lambda x: x['suspicion_score'], reverse=True)

        if problematic:
            print(f"\n🚨 MOST PROBLEMATIC FRAMES:")
            for i, frame in enumerate(problematic[:5]):
                print(f"   {i+1}. Frame {frame['frame_number']} @ {frame['timestamp']:.2f}s: {frame['classification']} (Score: {frame['suspicion_score']})")
                print(f"      Identity Similarity: {frame['identity_similarity']:.3f}")
                if frame['reasons']:
                    print(f"      Main Issue: {frame['reasons'][0]}")

        if results.get('html_report_path'):
            print(f"\n📄 DETAILED REPORT:")
            print(f"   Visual HTML report with frame images: {results['html_report_path']}")
            print(f"   Open this file in your browser for detailed frame-by-frame analysis")

    def _create_opencv_fallback(self):
        """Fallback when DeepFace is not available"""
        return {
            'error': 'DeepFace is required for expression-aware face analysis',
            'suggestion': 'Install DeepFace: pip install deepface',
            'note': 'Expression-aware analysis requires facial embeddings'
        }

26-05-04 04:54:39 - Directory /root/.deepface has been created
26-05-04 04:54:39 - Directory /root/.deepface/weights has been created


In [ ]:
# 📦 FULL DEEPFAKE ANALYSIS PIPELINE

def extract_optimal_deepfake_frames(video_path, max_frames=300, quality_focus="deepfake"):
    print("INITIALIZING OPTIMAL DEEPFAKE FRAME EXTRACTOR...")
    extractor = OptimalDeepfakeFrameExtractor()
    results = extractor.extract_optimal_frames(
        video_path=video_path,
        max_frames=max_frames,
        quality_focus=quality_focus
    )
    return results


def run_expression_aware_face_analysis(extracted_frames_result, max_frames=50):
    print("STARTING FACE ANALYSIS")
    print("=" * 60)

    detector = ExpressionAwareFaceWeightDetector()

    results = detector.analyze_expression_aware_face_patterns(
        extracted_frames_result,
        max_frames=max_frames
    )

    if results.get('html_report_path'):
        print(f"📄 Visual report available: {results['html_report_path']}")

    return results


def deepfake_expression_pipeline(
    max_frames_extraction=300,
    max_frames_expression=50,
    quality_focus="deepfake"
):
    """
    🔄 Upload a video in Colab, run:
      1. Optimal Deepfake Frame Extraction
      2. Expression-Aware Face Weight Analysis
    """

    from google.colab import files

    print("Please upload your video file...")
    uploaded = files.upload()

    video_path = list(uploaded.keys())[0]
    print(f"✅ Uploaded video: {video_path}")

    # STEP 1: Frame Extraction
    frame_results = extract_optimal_deepfake_frames(
        video_path=video_path,
        max_frames=max_frames_extraction,
        quality_focus=quality_focus
    )

    print(f"✅ Frame extraction complete: {len(frame_results['all_frames'])} frames extracted")

    # STEP 2: Expression-Aware Analysis
    expression_results = run_expression_aware_face_analysis(
        frame_results,
        max_frames=max_frames_expression
    )

    print("✅ face analysis complete!")

    return {
        "video_path": video_path,
        "frame_extraction_results": frame_results,
        "expression_analysis_results": expression_results
    }


results = deepfake_expression_pipeline(
    max_frames_extraction=250,
    max_frames_expression=40
)

Please upload your video file...


Saving fake video.mp4 to fake video.mp4
✅ Uploaded video: fake video.mp4
INITIALIZING OPTIMAL DEEPFAKE FRAME EXTRACTOR...
🎯 OPTIMAL DEEPFAKE FRAME EXTRACTOR INITIALIZED
   🎭 Face-priority extraction enabled
   ⚡ Dense temporal sampling (0.1-0.2s intervals)
   🔍 Motion transition detection active
   📊 Multi-scale quality assessment
   🎬 Optimized for modern deepfake detection
🎬 OPTIMAL DEEPFAKE ANALYSIS STARTING:
   📊 Resolution: 1920x1080
   ⏱️  Duration: 35.1 seconds
   🎬 FPS: 24.0
   📝 Total Frames: 843
   🎯 Target Extraction: 250 frames (7.1/sec)
   📈 Base sampling: every 3 frames (0.15s)

🔍 Starting intelligent frame extraction...
   🔄 Extracted 50/250 frames (0.4/sec, ETA: 475.0s)
   🔄 Extracted 100/250 frames (0.4/sec, ETA: 354.4s)
   🔄 Extracted 150/250 frames (0.4/sec, ETA: 235.7s)
   🔄 Extracted 200/250 frames (0.4/sec, ETA: 117.5s)
   🔄 Extracted 250/250 frames (0.4/sec, ETA: 0.0s)
   ✅ Extraction complete! 250 frames in 586.0s

 OPTIMAL DEEPFAKE ANALYSIS COMPLETE
📊 EXTRACTIO

Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5
To: /root/.deepface/weights/facenet_weights.h5


26-05-04 05:05:45 - 🔗 facenet_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5 to /root/.deepface/weights/facenet_weights.h5...


100%|██████████| 92.2M/92.2M [00:00<00:00, 145MB/s]


      ✅ Weights: 128D, norm=12.031, size=120409
   📊 Processing Frame 2 (2/40) @ 0.08s
      ✅ Weights: 128D, norm=12.076, size=121801
   📊 Processing Frame 3 (3/40) @ 0.12s
      ✅ Weights: 128D, norm=12.125, size=122500
   📊 Processing Frame 19 (4/40) @ 0.79s
      ✅ Weights: 128D, norm=12.100, size=126025
   📊 Processing Frame 25 (5/40) @ 1.04s
      ✅ Weights: 128D, norm=12.116, size=123201
   📊 Processing Frame 31 (6/40) @ 1.29s
      ✅ Weights: 128D, norm=4.722, size=127449
   📊 Processing Frame 37 (7/40) @ 1.54s
      ✅ Weights: 128D, norm=11.928, size=121801
   📊 Processing Frame 43 (8/40) @ 1.79s
      ✅ Weights: 128D, norm=12.196, size=116964
   📊 Processing Frame 49 (9/40) @ 2.04s
      ✅ Weights: 128D, norm=4.953, size=126736
   📊 Processing Frame 55 (10/40) @ 2.29s
      ✅ Weights: 128D, norm=11.886, size=126025
   📊 Processing Frame 61 (11/40) @ 2.54s
      ✅ Weights: 128D, norm=12.083, size=124609
   📊 Processing Frame 67 (12/40) @ 2.79s
      ✅ Weights: 128D, norm=11.74

In [ ]:
# # ==========================
# # 📦 CLEAN BATCH DEEPFAKE ANALYSIS PIPELINE
# # Process Multiple Videos for Deepfake Detection
# # ==========================

# import os
# import pandas as pd
# import time
# from datetime import datetime
# from pathlib import Path

# def batch_deepfake_analysis(
#     video_folder_path,
#     output_folder="batch_analysis_results",
#     max_frames_extraction=250,
#     max_frames_expression=40,
#     quality_focus="deepfake"
# ):
#     """
#     Process multiple videos for deepfake detection

#     Args:
#         video_folder_path: Path to folder containing videos
#         output_folder: Folder to save results
#         max_frames_extraction: Max frames for extraction phase
#         max_frames_expression: Max frames for face analysis phase
#         quality_focus: Focus mode for extraction

#     Returns:
#         dict: Analysis results with statistics and CSV file path
#     """

#     # Create output directory
#     os.makedirs(output_folder, exist_ok=True)

#     # Supported video formats
#     video_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv', '.webm', '.m4v'}

#     # Find all video files
#     video_files = []
#     for ext in video_extensions:
#         video_files.extend(Path(video_folder_path).glob(f"*{ext}"))
#         video_files.extend(Path(video_folder_path).glob(f"*{ext.upper()}"))

#     if not video_files:
#         print(f"❌ No video files found in {video_folder_path}")
#         return None

#     print(f"🎬 BATCH DEEPFAKE ANALYSIS INITIALIZED")
#     print(f"📂 Video folder: {video_folder_path}")
#     print(f"📊 Found {len(video_files)} video files")
#     print(f"💾 Results will be saved to: {output_folder}")
#     print("=" * 60)

#     # Results storage
#     batch_results = []
#     start_time = time.time()
#     processed_count = 0
#     failed_count = 0

#     # Process each video
#     for i, video_path in enumerate(video_files, 1):
#         video_name = video_path.name
#         print(f"\n🎥 Processing Video {i}/{len(video_files)}: {video_name}")

#         try:
#             # STEP 1: Frame Extraction
#             print(f"   📸 Extracting frames...")
#             extractor = OptimalDeepfakeFrameExtractor()
#             frame_results = extractor.extract_optimal_frames(
#                 video_path=str(video_path),
#                 max_frames=max_frames_extraction,
#                 quality_focus=quality_focus
#             )

#             # STEP 2: Expression-Aware Analysis
#             print(f"   🎭 Running face analysis...")
#             detector = ExpressionAwareFaceWeightDetector()
#             expression_results = detector.analyze_expression_aware_face_patterns(
#                 frame_results,
#                 max_frames=max_frames_expression
#             )

#             # Extract key results
#             if 'overall_assessment' in expression_results:
#                 verdict = expression_results['overall_assessment']['verdict']
#                 probability = expression_results['overall_assessment']['deepfake_probability']
#                 confidence = expression_results['overall_assessment']['confidence']
#                 frame_stats = expression_results['frame_statistics']

#                 # Classify into main categories
#                 classification = classify_video_result(verdict)

#                 video_result = {
#                     'video_name': video_name,
#                     'video_path': str(video_path),
#                     'classification': classification,
#                     'verdict': verdict,
#                     'deepfake_probability': probability,
#                     'confidence': confidence,
#                     'total_frames_analyzed': frame_stats['total_frames_analyzed'],
#                     'likely_fake_frames': frame_stats['likely_fake_frames'],
#                     'suspicious_frames': frame_stats['suspicious_frames'],
#                     'questionable_frames': frame_stats['questionable_frames'],
#                     'appears_real_frames': frame_stats['appears_real_frames'],
#                     'problematic_percentage': frame_stats['problematic_percentage'],
#                     'processing_time': expression_results['processing_stats']['processing_time'],
#                     'status': 'SUCCESS',
#                     'timestamp': datetime.now().isoformat()
#                 }

#                 print(f"   ✅ Result: {classification} ({probability:.1f}% deepfake probability)")
#                 processed_count += 1

#             else:
#                 # Handle analysis failure
#                 video_result = create_failed_result(video_name, str(video_path), "Analysis failed")
#                 print(f"   ❌ Analysis failed for {video_name}")
#                 failed_count += 1

#         except Exception as e:
#             print(f"   ❌ Error processing {video_name}: {str(e)}")
#             video_result = create_failed_result(video_name, str(video_path), str(e))
#             failed_count += 1

#         batch_results.append(video_result)

#     # Calculate final statistics
#     end_time = time.time()
#     total_time = end_time - start_time

#     # Generate and save final results
#     final_results = generate_final_report(
#         batch_results,
#         {
#             'total_videos': len(video_files),
#             'processed': processed_count,
#             'failed': failed_count,
#             'total_time': total_time
#         },
#         output_folder
#     )

#     print(f"\n🏆 BATCH ANALYSIS COMPLETE!")
#     print(f"📊 Processed: {processed_count}/{len(video_files)} videos")
#     print(f"❌ Failed: {failed_count} videos")
#     print(f"⏱️ Total time: {total_time:.1f}s")
#     print(f"📁 Results saved in: {output_folder}")

#     return final_results

# def classify_video_result(verdict):
#     """Classify verdict into main categories"""
#     if "LIKELY DEEPFAKE" in verdict:
#         return "LIKELY DEEPFAKE"
#     elif "SUSPICIOUS" in verdict:
#         return "SUSPICIOUS CONTENT"
#     elif "INVESTIGATION" in verdict:
#         return "NEEDS INVESTIGATION"
#     elif "AUTHENTIC" in verdict:
#         return "LIKELY AUTHENTIC"
#     else:
#         return "NEEDS INVESTIGATION"

# def create_failed_result(video_name, video_path, error_msg):
#     """Create result entry for failed analysis"""
#     return {
#         'video_name': video_name,
#         'video_path': video_path,
#         'classification': 'ANALYSIS FAILED',
#         'verdict': f'Error: {error_msg}',
#         'deepfake_probability': None,
#         'confidence': 'N/A',
#         'total_frames_analyzed': 0,
#         'likely_fake_frames': 0,
#         'suspicious_frames': 0,
#         'questionable_frames': 0,
#         'appears_real_frames': 0,
#         'problematic_percentage': 0,
#         'processing_time': 0,
#         'status': 'FAILED',
#         'error': error_msg,
#         'timestamp': datetime.now().isoformat()
#     }

# def generate_final_report(results, stats, output_folder):
#     """Generate final CSV report with statistics"""

#     # Create DataFrame
#     df = pd.DataFrame(results)

#     # Calculate classification statistics
#     total_videos = len(results)
#     successful_analyses = len([r for r in results if r['status'] == 'SUCCESS'])

#     # Count each classification type
#     likely_deepfakes = len([r for r in results if r['classification'] == 'LIKELY DEEPFAKE'])
#     suspicious_content = len([r for r in results if r['classification'] == 'SUSPICIOUS CONTENT'])
#     needs_investigation = len([r for r in results if r['classification'] == 'NEEDS INVESTIGATION'])
#     likely_authentic = len([r for r in results if r['classification'] == 'LIKELY AUTHENTIC'])
#     analysis_failed = len([r for r in results if r['classification'] == 'ANALYSIS FAILED'])

#     # Calculate percentages
#     deepfake_percentage = (likely_deepfakes / total_videos) * 100 if total_videos > 0 else 0
#     suspicious_percentage = (suspicious_content / total_videos) * 100 if total_videos > 0 else 0
#     problematic_percentage = ((likely_deepfakes + suspicious_content) / total_videos) * 100 if total_videos > 0 else 0
#     authentic_percentage = (likely_authentic / total_videos) * 100 if total_videos > 0 else 0

#     # Create timestamp for files
#     timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

#     # Save main results CSV
#     results_csv_file = os.path.join(output_folder, f"deepfake_analysis_results_{timestamp}.csv")
#     df.to_csv(results_csv_file, index=False)

#     # Create summary statistics
#     summary_data = {
#         'Analysis Summary': [
#             f"Total Videos Analyzed: {total_videos}",
#             f"Successfully Processed: {successful_analyses}",
#             f"Analysis Failed: {analysis_failed}",
#             "",
#             "=== DETECTION RESULTS ===",
#             f"Likely Deepfakes: {likely_deepfakes} ({deepfake_percentage:.1f}%)",
#             f"Suspicious Content: {suspicious_content} ({suspicious_percentage:.1f}%)",
#             f"Needs Investigation: {needs_investigation}",
#             f"Likely Authentic: {likely_authentic} ({authentic_percentage:.1f}%)",
#             "",
#             f"🚨 TOTAL PROBLEMATIC CONTENT: {likely_deepfakes + suspicious_content} ({problematic_percentage:.1f}%)",
#             "",
#             f"Processing Time: {stats['total_time']:.1f} seconds",
#             f"Average Time per Video: {stats['total_time']/max(successful_analyses, 1):.1f} seconds"
#         ]
#     }

#     # Save summary CSV
#     summary_df = pd.DataFrame(summary_data)
#     summary_csv_file = os.path.join(output_folder, f"analysis_summary_{timestamp}.csv")
#     summary_df.to_csv(summary_csv_file, index=False)

#     # Print detailed summary
#     print(f"\n📊 FINAL ANALYSIS SUMMARY:")
#     print(f"   📁 Main Results: {results_csv_file}")
#     print(f"   📋 Summary Report: {summary_csv_file}")

#     print(f"\n🎯 DETECTION BREAKDOWN:")
#     print(f"   🚨 Likely Deepfakes: {likely_deepfakes} videos ({deepfake_percentage:.1f}%)")
#     print(f"   ⚠️  Suspicious Content: {suspicious_content} videos ({suspicious_percentage:.1f}%)")
#     # print(f"   🔍 Needs Investigation: {needs_investigation} videos")
#     print(f"   ✅ Likely Authentic: {likely_authentic} videos ({authentic_percentage:.1f}%)")
#     print(f"   ❌ Analysis Failed: {analysis_failed} videos")

#     print(f"\n🚨 TOTAL PROBLEMATIC CONTENT: {likely_deepfakes + suspicious_content} videos ({problematic_percentage:.1f}%)")

#     # Auto-download in Colab environment
#     try:
#         from google.colab import files
#         print(f"\n📥 Auto-downloading results...")
#         files.download(results_csv_file)
#         files.download(summary_csv_file)
#         print(f"✅ Files downloaded successfully!")
#     except ImportError:
#         print(f"\n💡 CSV files saved locally. Download manually if needed.")

#     return {
#         'results_csv': results_csv_file,
#         'summary_csv': summary_csv_file,
#         'statistics': {
#             'total_videos': total_videos,
#             'successful_analyses': successful_analyses,
#             'likely_deepfakes': likely_deepfakes,
#             'deepfake_percentage': deepfake_percentage,
#             'suspicious_content': suspicious_content,
#             'suspicious_percentage': suspicious_percentage,
#             'problematic_content': likely_deepfakes + suspicious_content,
#             'problematic_percentage': problematic_percentage,
#             'likely_authentic': likely_authentic,
#             'authentic_percentage': authentic_percentage,
#             'analysis_failed': analysis_failed
#         },
#         'all_results': results
#     }

# # ==========================
# # 🚀 USAGE FUNCTION
# # ==========================

# def run_batch_analysis(video_folder_path):
#     """
#     Main function to run batch deepfake analysis

#     Usage:
#         video_folder = "/path/to/your/video/folder"
#         results = run_batch_analysis(video_folder)
#     """
#     return batch_deepfake_analysis(
#         video_folder_path=video_folder_path,
#         output_folder="deepfake_batch_results",
#         max_frames_extraction=250,
#         max_frames_expression=40,
#         quality_focus="deepfake"
#     )

# # ==========================
# # 📊 EXAMPLE USAGE
# # ==========================

# # Upload your video folder to Colab, then run:
# video_folder = "/content/drive/MyDrive/Authentic"  # Update this path
# results = run_batch_analysis(video_folder)

# # The function will:
# # 1. Process all videos in the folder
# # 2. Generate a detailed CSV with all results
# # 3. Create a summary CSV with statistics
# # 4. Show percentage of deepfakes detected
# # 5. Auto-download the CSV files in Colab

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# import os

# videos_folder = "/content/drive/MyDrive/Authentic"
# video_files = [os.path.join(videos_folder, f) for f in os.listdir(videos_folder) if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))]

# print(f"Found {len(video_files)} videos")
